In [2]:
from data_frame.transformation.filter.filter_builder import FilterBuilder
from pyspark.sql import types as T
from pyspark.sql import functions as F
from data_frame.spark_utils import get_spark

In [3]:
spark = get_spark(app_name="Schema Inspection")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/18 08:29:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
data = [
    (1, "Laptop", 1200, 4.5),
    (2, "Mouse", 25, 4.8),
    (3, "Keyboard", 75, 4.3),
    (4, "Monitor", 350, 4.6),
    (5, "Desk", 450, 4.2)
]
df = spark.createDataFrame(data, ["id", "product", "price", "rating"])

## 1. Range and Numeric Filters

In [5]:
# Price range filter
price_filtered = FilterBuilder.range_filter(df, "price", min_val=50, max_val=500)
print("Price between 50 and 500:")
price_filtered.show()


Price between 50 and 500:


+---+--------+-----+------+
| id| product|price|rating|
+---+--------+-----+------+
|  3|Keyboard|   75|   4.3|
|  4| Monitor|  350|   4.6|
|  5|    Desk|  450|   4.2|
+---+--------+-----+------+



## 2. Text Search Filters

In [10]:
# Partial text search
text_search = FilterBuilder.text_search(df, "product", "top", partial_match=True)
print("Products containing 'top':")
text_search.show()

# Exact match
exact_match = FilterBuilder.text_search(df, "product", "Mouse", partial_match=False)
print("Exact match for 'Mouse':")
exact_match.show()

Products containing 'top':
+---+-------+-----+------+
| id|product|price|rating|
+---+-------+-----+------+
|  1| Laptop| 1200|   4.5|
+---+-------+-----+------+

Exact match for 'Mouse':
+---+-------+-----+------+
| id|product|price|rating|
+---+-------+-----+------+
|  2|  Mouse|   25|   4.8|
+---+-------+-----+------+



## 3. List and Null Filters

In [9]:
# Filter by list
list_filtered = FilterBuilder.filter_by_list(
    df, "product", ["Laptop", "Monitor", "Mouse"]
)
print("Products in list:")
list_filtered.show()

Products in list:
+---+-------+-----+------+
| id|product|price|rating|
+---+-------+-----+------+
|  1| Laptop| 1200|   4.5|
|  2|  Mouse|   25|   4.8|
|  4|Monitor|  350|   4.6|
+---+-------+-----+------+



In [8]:
# Exclude list
exclude_filtered = FilterBuilder.filter_by_list(
    df, "product", ["Desk"], exclude=True
)
print("Exclude 'Desk':")
exclude_filtered.show()

Exclude 'Desk':
+---+--------+-----+------+
| id| product|price|rating|
+---+--------+-----+------+
|  1|  Laptop| 1200|   4.5|
|  2|   Mouse|   25|   4.8|
|  3|Keyboard|   75|   4.3|
|  4| Monitor|  350|   4.6|
+---+--------+-----+------+



In [7]:

"""
## 4. Complex Conditions
"""
# Multiple conditions
conditions = [
    F.col("price") > 100,
    F.col("rating") > 4.4,
    ~F.col("product").isin(["Mouse", "Keyboard"])
]
complex_filtered = FilterBuilder.complex_and_filter(df, conditions)
print("Complex filter (price>100, rating>4.4, not Mouse/Keyboard):")
complex_filtered.show()

Complex filter (price>100, rating>4.4, not Mouse/Keyboard):
+---+-------+-----+------+
| id|product|price|rating|
+---+-------+-----+------+
|  1| Laptop| 1200|   4.5|
|  4|Monitor|  350|   4.6|
+---+-------+-----+------+

